# nsight-matmul-kfp

KFP project on the Miramar platform.

- Define pipeline components in `pipeline.py`
- Use the cells below to compile, inspect, and submit runs locally
- Use **Deploy to KFP** (GitHub Actions) for CI/CD submission


In [ ]:
import kfp
from kfp import compiler, dsl
print(f'kfp {kfp.__version__}')

In [ ]:
# Compile the pipeline defined in pipeline.py
from pipeline import pipeline
compiler.Compiler().compile(pipeline_func=pipeline, package_path='/tmp/pipeline.yaml')
print('Compiled → /tmp/pipeline.yaml')

In [ ]:
# Connect to KFP (requires SSH tunnel: ssh -L 8080:localhost:8080 spark-79b7.local)
client = kfp.Client(host='http://localhost:8080')
client.list_pipelines()

In [ ]:
# Submit a run
run = client.create_run_from_pipeline_package(
    pipeline_file='/tmp/pipeline.yaml',
    arguments={},
    run_name='notebook-run',
)
print(f'Run ID: {run.run_id}')
print(f'UI: http://localhost:8080/#/runs/details/{run.run_id}')

In [ ]:
# Poll run status
import time
run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f'  {state}')
    if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(10)